# Mind the Map · C1: auditing the released code graphs (v0)

This notebook reruns our audit of the competition's released code graphs against Python-AST ground truth
from the task snapshots. It uses the same `mtm/` source files as our repository.

- **Table 1** below is recomputed here from `/kaggle/input`, then checked against our local run.
- **The validity tables** at the end come from a local Docker sweep. They are copied in, not rerun,
  because notebooks have no Docker.

In [ ]:
import os, glob
from pathlib import Path
hits = glob.glob("/kaggle/input/**/tasks.jsonl", recursive=True)
assert hits, "attach the gemma-4-developer-agent competition data"
DATA = Path(hits[0]).parent
os.environ["MTM_DATA"] = str(DATA)
os.environ["MTM_OUT"] = "/kaggle/working/audit"
os.makedirs("mtm", exist_ok=True)
open("mtm/__init__.py", "w").close()
print(DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
%%writefile mtm/ast_index.py
"""Python-AST ground truth for a repo snapshot, in the released graph's node-id format.

Node ids in the released graphs are dotted module paths from the repo root with a leading
`src/` dropped and `__init__` collapsed to the package, then `Class.method` qualifiers,
e.g. `requests.sessions.Session.request` for `src/requests/sessions.py`.
"""
import ast
import tarfile
from dataclasses import dataclass, field
from pathlib import Path

KINDS = ("module", "class", "function", "async_function")


@dataclass
class Symbol:
    id: str
    kind: str  # one of KINDS
    path: str  # repo-relative file path
    start: int
    end: int
    nested: bool = False  # defined inside a function body (not reachable as an attribute)
    is_test: bool = False
    flat_id: str = ""  # the id the released builder gives this symbol (see _walk)
    def_line: int = 0  # line of the `def`/`class` keyword (after decorators)


@dataclass
class RepoIndex:
    symbols: dict[str, Symbol] = field(default_factory=dict)
    by_path: dict[str, list[Symbol]] = field(default_factory=dict)
    sources: dict[str, str] = field(default_factory=dict)
    parse_errors: list[str] = field(default_factory=list)

    def enclosing(self, path: str, line: int) -> Symbol | None:
        """Innermost non-module symbol whose span contains `line`, else the module."""
        best = None
        for s in self.by_path.get(path, ()):
            if s.start <= line <= s.end and (best is None or s.end - s.start <= best.end - best.start):
                best = s
        return best


def released_match(idx: "RepoIndex", s: Symbol, node_text: dict[str, str]) -> bool:
    """True iff the released node with this symbol's id holds this symbol's own source,
    not another symbol's that collides with it after flattening."""
    text = node_text.get(s.flat_id)
    if text is None:
        return False
    lines = idx.sources[s.path].splitlines()
    head = _strip_comment(lines[s.def_line - 1]).strip() if 0 < s.def_line <= len(lines) else ""
    # Released class texts carry their decorator lines and drop trailing comments.
    body = text.lstrip().splitlines()
    while body and body[0].lstrip().startswith("@"):
        body.pop(0)
    return "\n".join(body).lstrip().startswith(head) if head else True


def _strip_comment(line: str) -> str:
    """Drop a trailing `# ...` comment that is outside string literals."""
    quote = None
    for i, ch in enumerate(line):
        if quote:
            if ch == quote:
                quote = None
        elif ch in "'\"":
            quote = ch
        elif ch == "#":
            return line[:i]
    return line


def module_id(path: str, strip: int | None = None) -> str:
    """Dotted id; `strip` leading dirs are dropped (default: just a leading `src/`)."""
    parts = path.removesuffix(".py").split("/")
    if strip is None:
        strip = 1 if parts[0] == "src" else 0
    parts = parts[strip:]
    if parts and parts[-1] == "__init__":
        parts = parts[:-1]
    return ".".join(parts)


def is_test_path(path: str) -> bool:
    name = path.rsplit("/", 1)[-1]
    return path.startswith(("tests/", "test/")) or "/tests/" in path or name.startswith("test_") or name == "conftest.py"


def _walk(node: ast.AST, prefix: str, mod: str, path: str, in_func: bool, out: list[Symbol], test: bool,
          state: dict) -> None:
    """Emit symbols with their true qualified id and the id the released builder gives them.

    The released builder keeps a single `current_class`: a class sets it on entry and clears it to
    None on exit instead of restoring the outer class. Classes are always filed at module scope, and
    functions under `current_class` if set. So a class nested in a method makes the outer class's
    later methods module-level, and nested defs are flattened. `flat_id` reproduces that.
    """
    for child in ast.iter_child_nodes(node):
        if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and any(
                (d.id if isinstance(d, ast.Name) else getattr(d, "attr", "")) == "overload"
                for d in child.decorator_list):
            continue  # typing stub; the implementation that follows is the real definition
        if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            kind = ("class" if isinstance(child, ast.ClassDef)
                    else "async_function" if isinstance(child, ast.AsyncFunctionDef) else "function")
            start = min([d.lineno for d in child.decorator_list] + [child.lineno])
            sid = f"{prefix}.{child.name}"
            if kind == "class":
                flat = f"{mod}.{child.name}"
                state["cls"] = child.name
            else:
                flat = f"{mod}.{state['cls']}.{child.name}" if state["cls"] else f"{mod}.{child.name}"
            out.append(Symbol(sid, kind, path, start, child.end_lineno or start, in_func, test, flat, child.lineno))
            _walk(child, sid, mod, path, in_func or kind != "class", out, test, state)
            if kind == "class":
                state["cls"] = None
        else:
            _walk(child, prefix, mod, path, in_func, out, test, state)


def _file_symbols(path: str, tree: ast.AST, n_lines: int, strip: int | None) -> list[Symbol]:
    mid = module_id(path, strip)
    test = is_test_path(path)
    syms = [Symbol(mid, "module", path, 1, max(1, n_lines), False, test, mid, 1)]
    _walk(tree, mid, mid, path, False, syms, test, {"cls": None})
    return syms


def index_source(path: str, src: str, idx: RepoIndex, released: set[str] | None = None) -> None:
    """Index one file. With `released` ids, pick the leading-dir strip that best matches them:
    the released builder roots some trees below the repo root (e.g. `docs_src/x/y.py` -> `x.y`)."""
    idx.sources[path] = src
    try:
        tree = ast.parse(src)
    except (SyntaxError, ValueError):
        idx.parse_errors.append(path)
        return
    n = len(src.splitlines())
    syms = _file_symbols(path, tree, n, None)
    if released is not None:
        depth = path.count("/")
        cands = [_file_symbols(path, tree, n, k) for k in range(depth + 1)]
        best = max(cands, key=lambda c: sum(s.flat_id in released for s in c))
        if any(s.flat_id in released for s in best):
            syms = best
    idx.by_path[path] = syms
    for s in syms:
        idx.symbols.setdefault(s.id, s)  # first definition wins (e.g. if/else redefinitions)


def index_snapshot(tgz: Path, released: set[str] | None = None) -> RepoIndex:
    idx = RepoIndex()
    with tarfile.open(tgz) as tar:
        for m in tar:
            name = m.name.removeprefix("./")
            if not m.isfile() or not name.endswith(".py") or name.startswith(".git/"):
                continue
            f = tar.extractfile(m)
            if f is not None:
                index_source(name, f.read().decode("utf-8", errors="replace"), idx, released)
    return idx

In [ ]:
%%writefile mtm/audit.py
"""C1 Auditor: measure the released code graphs against Python-AST ground truth.

Outputs (results/audit/):
  coverage.csv    one row per (commit, kind, nested, is_test): ast count, released count
  gold_sites.csv  one row per (task, gold edit site): enclosing symbol, kind, in released graph
  reach.csv       one row per task: seeds, gold symbols, hop distance to nearest gold symbol
  edges.csv       one row per commit: released edge types and how many `calls` edges are really containment
  table1.md       the per-repo summary

Run from app/: ../.venv/bin/python -m mtm.audit [graph_dir]
"""
import csv
import json
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

import networkx as nx

from mtm.ast_index import RepoIndex, index_snapshot, released_match

DATA = Path(os.environ.get("MTM_DATA", Path(__file__).resolve().parents[1] / "data"))
OUT = Path(os.environ.get("MTM_OUT", Path(__file__).resolve().parents[1] / "results" / "audit"))
MAX_HOPS = 3
# Pre-registered seed rule: an issue token seeds every node whose last id component equals it,
# ignoring tokens shorter than 3 chars and tokens that match more than MAX_SEED_FANOUT nodes.
MAX_SEED_FANOUT = 50
IDENT = re.compile(r"[A-Za-z_][A-Za-z0-9_]{2,}")
HUNK = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")


def repo_short(task: dict) -> str:
    return task["instance_id"].rsplit("_", 1)[0]


def load_graph(graph_dir: Path, task: dict) -> tuple[nx.MultiDiGraph, dict[str, str]]:
    data = json.loads((graph_dir / f"{repo_short(task)}_{task['base_commit']}.json").read_text())
    return nx.node_link_graph(data, edges="edges"), {n["id"]: n.get("text") or "" for n in data["nodes"]}


def gold_lines(patch: str) -> dict[str, set[int]]:
    """Old-side line numbers each file's patch touches. Pure insertions map to the line above."""
    out: dict[str, set[int]] = defaultdict(set)
    path, old = None, 0
    for line in patch.splitlines():
        if line.startswith("--- "):
            src = line[4:].split("\t")[0].strip()
            path = None if src == "/dev/null" else src.removeprefix("a/")
        elif line.startswith("+++ "):
            if path is None:  # new file: no base symbol
                out[line[4:].split("\t")[0].strip().removeprefix("b/")].add(0)
        elif m := HUNK.match(line):
            old = int(m.group(1))
        elif path is None or line.startswith("\\"):
            continue
        elif line.startswith("-"):
            out[path].add(old)
            old += 1
        elif line.startswith("+"):
            out[path].add(max(1, old - 1))
        else:
            old += 1
    return out


def gold_symbols(idx: RepoIndex, patch: str, node_text: dict[str, str]) -> list[dict]:
    sites = {}
    for path, lines in gold_lines(patch).items():
        if not path.endswith(".py"):
            continue
        if path not in idx.by_path:
            sites[(path, None)] = dict(path=path, symbol="", kind="new_file", nested=False, flat_id="",
                                       in_released=False)
            continue
        for ln in sorted(lines):
            s = idx.enclosing(path, ln)
            if s is not None:
                sites[(path, s.id)] = dict(path=path, symbol=s.id, kind=s.kind, nested=s.nested, flat_id=s.flat_id,
                                           in_released=released_match(idx, s, node_text))
    return list(sites.values())


def seed_nodes(g: nx.MultiDiGraph, text: str) -> set[str]:
    by_name = defaultdict(set)
    for n in g.nodes:
        by_name[n.rsplit(".", 1)[-1]].add(n)
    seeds = set()
    for tok in set(IDENT.findall(text)):
        hits = by_name.get(tok, ())
        if 0 < len(hits) <= MAX_SEED_FANOUT:
            seeds |= hits
    return seeds


def hop_distance(g: nx.MultiDiGraph, seeds: set[str], targets: set[str]) -> int | None:
    """Undirected hops from any seed to any target, or None if not within MAX_HOPS."""
    if not seeds or not targets:
        return None
    dist = nx.multi_source_dijkstra_path_length(g.to_undirected(as_view=True), seeds, cutoff=MAX_HOPS)
    hits = [d for t, d in dist.items() if t in targets]
    return int(min(hits)) if hits else None


def edge_stats(g: nx.MultiDiGraph) -> dict:
    types = Counter(d.get("type", "") for _, _, d in g.edges(data=True))
    containment = sum(1 for u, v, d in g.edges(data=True)
                      if d.get("type") == "calls" and v.startswith(u + ".") and "." not in v[len(u) + 1:])
    return dict(edges=g.number_of_edges(), types=dict(types), calls_that_are_containment=containment)


def write_csv(path: Path, rows: list[dict]) -> None:
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)


def main(graph_dir: str = str(DATA / "graphs")) -> None:
    gdir = Path(graph_dir)
    OUT.mkdir(parents=True, exist_ok=True)
    tasks = [json.loads(l) for l in (DATA / "tasks.jsonl").open()]
    by_commit = defaultdict(list)
    for t in tasks:
        by_commit[(repo_short(t), t["base_commit"])].append(t)

    cov, gold, reach, edges = [], [], [], []
    for n, ((repo, commit), group) in enumerate(sorted(by_commit.items()), 1):
        g, node_text = load_graph(gdir, group[0])
        released = set(g.nodes)
        idx = index_snapshot(DATA / "snapshots" / f"{group[0]['instance_id']}.tgz", released)
        explained = {s.flat_id for s in idx.symbols.values()}
        flat_counts = Counter(s.flat_id for s in idx.symbols.values() if s.kind != "module")

        c = Counter()
        for s in idx.symbols.values():
            c[(s.kind, s.nested, s.is_test, "ast")] += 1
            c[(s.kind, s.nested, s.is_test, "released")] += released_match(idx, s, node_text)
        for (kind, nested, test, col), v in c.items():
            if col == "ast":
                cov.append(dict(repo=repo, commit=commit, kind=kind, nested=nested, is_test=test,
                                ast=v, released=c[(kind, nested, test, "released")]))
        edges.append(dict(repo=repo, commit=commit, nodes=len(released),
                          unexplained_nodes=len(released - explained), parse_errors=len(idx.parse_errors),
                          misqualified=sum(1 for s in idx.symbols.values()
                                           if s.kind != "module" and not s.nested and s.flat_id != s.id),
                          id_collisions=sum(v - 1 for v in flat_counts.values() if v > 1),
                          **{k: json.dumps(v) if isinstance(v, dict) else v for k, v in edge_stats(g).items()}))

        for t in group:
            sites = gold_symbols(idx, t["patch"], node_text)
            for s in sites:
                gold.append(dict(instance_id=t["instance_id"], repo=repo, **s))
            targets = {s["flat_id"] for s in sites if s["in_released"]}
            seeds = seed_nodes(g, t["problem_statement"])
            reach.append(dict(instance_id=t["instance_id"], repo=repo, n_seeds=len(seeds),
                              n_gold=len(sites), n_gold_in_graph=len(targets),
                              hops=hop_distance(g, seeds, targets)))
        print(f"[{n}/{len(by_commit)}] {repo} {commit[:8]} nodes={len(released)} tasks={len(group)}", flush=True)

    write_csv(OUT / "coverage.csv", cov)
    write_csv(OUT / "gold_sites.csv", gold)
    write_csv(OUT / "reach.csv", reach)
    write_csv(OUT / "edges.csv", edges)
    (OUT / "table1.md").write_text(table1(cov, gold, reach, edges))
    print((OUT / "table1.md").read_text())


def pct(a: int, b: int) -> str:
    return f"{a}/{b} ({100 * a / b:.1f}%)" if b else "–"


def table1(cov, gold, reach, edges) -> str:
    repos = sorted({r["repo"] for r in reach}) + ["all"]
    pick = lambda rows, r: [x for x in rows if r == "all" or x["repo"] == r]
    lines = ["| repo | tasks | commits | top-level sync fn | top-level async fn | classes | modules | nested defs | "
             "edge types | `calls` that are containment | top-level defs mis-qualified | id collisions | gold sites in graph | async gold sites in graph | "
             "gold reachable ≤1 hop | ≤2 hops | ≤3 hops |", "|" + "---|" * 17]
    for r in repos:
        cv, gd, rc, ed = pick(cov, r), pick(gold, r), pick(reach, r), pick(edges, r)
        s = lambda kind, nested: (sum(x["released"] for x in cv if x["kind"] == kind and x["nested"] == nested),
                                  sum(x["ast"] for x in cv if x["kind"] == kind and x["nested"] == nested))
        nested = [s(k, True) for k in ("function", "async_function", "class")]
        types = Counter()
        for e in ed:
            types.update(json.loads(e["types"]))
        n_edges = sum(e["edges"] for e in ed)
        sites = [x for x in gd if x["kind"] != "module"]
        asy = [x for x in sites if x["kind"] == "async_function"]
        reach_k = lambda k: pct(sum(1 for x in rc if x["hops"] is not None and x["hops"] <= k), len(rc))
        lines.append("| " + " | ".join([
            r, str(len(rc)), str(len(ed)),
            pct(*s("function", False)), pct(*s("async_function", False)), pct(*s("class", False)),
            pct(*s("module", False)), pct(sum(a for a, _ in nested), sum(b for _, b in nested)),
            ", ".join(f"{k}:{v}" for k, v in types.items()),
            pct(sum(e["calls_that_are_containment"] for e in ed), n_edges),
            str(sum(e["misqualified"] for e in ed)), str(sum(e["id_collisions"] for e in ed)),
            pct(sum(x["in_released"] for x in sites), len(sites)),
            pct(sum(x["in_released"] for x in asy), len(asy)),
            reach_k(1), reach_k(2), reach_k(3)]) + " |")
    return "\n".join(lines) + "\n"


if __name__ == "__main__":
    main(*sys.argv[1:])

## Run the audit (~2 min)

In [ ]:
from mtm import audit
audit.main(str(DATA / "graphs"))

## Table 1 matches the local run

In [ ]:
from IPython.display import Markdown, display
EXPECTED = '| repo | tasks | commits | top-level sync fn | top-level async fn | classes | modules | nested defs | edge types | `calls` that are containment | top-level defs mis-qualified | id collisions | gold sites in graph | async gold sites in graph | gold reachable ≤1 hop | ≤2 hops | ≤3 hops |\n|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|\n| fastapi | 67 | 67 | 204952/204952 (100.0%) | 0/65317 (0.0%) | 52689/52710 (100.0%) | 0/79148 (0.0%) | 11384/17744 (64.2%) | calls:164682 | 13991/164682 (8.5%) | 1279 | 6752 | 324/401 (80.8%) | 0/51 (0.0%) | 17/67 (25.4%) | 17/67 (25.4%) | 27/67 (40.3%) |\n| httpx | 1 | 1 | 661/661 (100.0%) | 0/98 (0.0%) | 78/78 (100.0%) | 0/45 (0.0%) | 1/3 (33.3%) | calls:1618 | 482/1618 (29.8%) | 0 | 0 | 14/19 (73.7%) | 0/5 (0.0%) | 1/1 (100.0%) | 1/1 (100.0%) | 1/1 (100.0%) |\n| requests | 13 | 12 | 7214/7214 (100.0%) | – | 869/869 (100.0%) | 0/433 (0.0%) | 982/994 (98.8%) | calls:21896 | 4533/21896 (20.7%) | 1425 | 255 | 13/13 (100.0%) | – | 5/13 (38.5%) | 6/13 (46.2%) | 7/13 (53.8%) |\n| rich | 48 | 47 | 75969/76439 (99.4%) | – | 10174/10174 (100.0%) | 0/9231 (0.0%) | 7489/8136 (92.0%) | calls:264392 | 40471/264392 (15.3%) | 0 | 2104 | 117/140 (83.6%) | – | 17/48 (35.4%) | 22/48 (45.8%) | 29/48 (60.4%) |\n| all | 129 | 127 | 288796/289266 (99.8%) | 0/65415 (0.0%) | 63810/63831 (100.0%) | 0/88857 (0.0%) | 19856/26877 (73.9%) | calls:452588 | 59477/452588 (13.1%) | 2704 | 9111 | 468/573 (81.7%) | 0/56 (0.0%) | 40/129 (31.0%) | 46/129 (35.7%) | 64/129 (49.6%) |\n'
got = Path("/kaggle/working/audit/table1.md").read_text()
display(Markdown(got))
print("identical to local run:", got == EXPECTED)
assert got == EXPECTED

## Gold-patch validity (local Docker sweep, copied in, not rerun here)

A task is *verifiable* when the reference (gold) patch resolves and an empty patch does not.

**Official public kit**

| repo | tasks | verifiable | gold fails | empty passes | no empty run |
|---|---|---|---|---|---|
| fastapi | 67 | 1 | 66 | 0 | 0 |
| httpx | 1 | 0 | 1 | 0 | 0 |
| requests | 13 | 0 | 10 | 3 | 0 |
| rich | 48 | 38 | 9 | 1 | 0 |
| all | 129 | 39 | 86 | 4 | 0 |

Gold-patch failure reasons:

- env: missing wheel `typing_inspection`: 62
- tests fail with gold patch: 9
- env: missing wheel `pytest-httpbin`: 8
- env: missing wheel `dirty_equals`: 4
- env: AttributeError: module 'httpx' has no attribute 'Stream': 1
- env: missing wheel `attr`: 1
- harness: Required test node did not pass: test_traceback_finely_grained_missing: 1


**Documented `wheels_plus` setup:** the official wheels, 12 extra wheels, and a step that installs each repo's
pinned dependency versions. Hashes and harness findings are in our repository.

| repo | tasks | verifiable | gold fails | empty passes | no empty run |
|---|---|---|---|---|---|
| fastapi | 67 | 60 | 5 | 2 | 0 |
| httpx | 1 | 0 | 1 | 0 | 0 |
| requests | 13 | 0 | 10 | 3 | 0 |
| rich | 48 | 38 | 9 | 1 | 0 |
| all | 129 | 98 | 25 | 6 | 0 |

Gold-patch failure reasons:

- tests fail with gold patch: 22
- harness: Required test node did not pass: test_model_description_escaped_with_f: 1
- env: AttributeError: module 'httpx' has no attribute 'Stream': 1
- harness: Required test node did not pass: test_traceback_finely_grained_missing: 1


⚠️ Kaggle's scoring environment may differ from both local setups.